In [1]:
# importar todas las paqueterias necesarias 
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import squarify

In [2]:
# son muchos datos y es necesario que las columnas sean visibles 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
# las columnas a veces traen información muy extensa que es importante ver
pd.set_option('display.max_colwidth', None)

In [4]:
# el diccionario de datos nos permite saber el significado de las columnas
# lo convierto en Data Frame para que sea más fácil de visualizar
# el encoding es por los caracteres especiales del español y la memoria porque es extenso
# este diccionario viene en partes
diccionarioP3= pd.read_csv('Datos (CNGSPSPE) 2020/diccionario_de_datos_m3s1p3_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
diccionarioP3

,COLUMNA,DESCRIPCION,ORIGEN,TIPO_DATO,LONGITUD,CODIGO_VALIDO
0,entidad_a,Entidad federativa,SECC I. 3,Numérico,2,1 al 99
1,numeral,Identificador o numeral del orden que guarda cada registro capturado dentro de la lista,SECC I. 3,Carácter,4,0-5000
2,nombre1,Nombre de los centros penitenciarios,SECC I. 3,Carácter,500,NaN
3,rensnn1,Operación bajo el esquema de contrato de prestación de servicios,SECC I. 3,Numérico,2,0 al 98
4,anio,Año en el que entró en operación el esquema de contrato de prestación de servicios,SECC I. 3,NaN,4,NaN


Segun este diccionario la variable que nos interesa es rensnn1 para saber si los centros penitenciarios operan bajo el esquema d econtrato de prestación de servicios 

In [5]:
# valores de rensnn1
valores_rensnn1= pd.read_csv('Datos (CNGSPSPE) 2020/rensnn1_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
valores_rensnn1

,rensnn1,descrip
0,0,No aplica
1,1,Sí
2,2,No
3,9,No se sabe
4,97,No cuenta con elementos para responder
5,98,Información no disponible


In [6]:
# df = Data Frame
df_exploratorioP3= pd.read_csv('Datos (CNGSPSPE) 2020/m3s1p3_cngspspe2020.csv',
    encoding='utf-8-sig',
    low_memory=False
)
df_exploratorioP3['rensnn1'].value_counts()

rensnn1
2    220
1     29
9      5
Name: count, dtype: int64

De acuerdo con esto, hay 29 centros que operan bajo esta figura y 5 que no se sabe, veamos cuale son

In [7]:
df_exploratorioP3[df_exploratorioP3['rensnn1']!=2]

,entidad_a,numeral,nombre1,rensnn1,anio
4,2,1,CENTRO DE REINSERCIÓN SOCIAL MEXICALI,9,NSS
5,2,2,CENTRO DE REINSERCIÓN SOCIAL TIJUANA,9,NSS
6,2,3,CENTRO DE REINSERCIÓN SOCIAL ENSENADA,9,NSS
7,2,4,CENTRO DE REINSERCIÓN SOCIAL EL HONGO,9,NSS
8,2,5,CENTRO DE REINSERCIÓN SOCIAL EL HONGO II,9,NSS
56,9,11,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,1,2015
57,9,12,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,1,2015
59,10,1,CERESO NO. 1,1,1970
60,10,2,CE.DI.RE.SO. NO. 1,1,2001
61,10,3,CE.DI.RE.SO. NO. 2,1,2007


Hagamos un Data frame con solo los que sabemos que son CPS 

In [8]:
Df_CPS = df_exploratorioP3[df_exploratorioP3['rensnn1']==1]
Df_CPS

,entidad_a,numeral,nombre1,rensnn1,anio
56,9,11,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,1,2015
57,9,12,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,1,2015
59,10,1,CERESO NO. 1,1,1970
60,10,2,CE.DI.RE.SO. NO. 1,1,2001
61,10,3,CE.DI.RE.SO. NO. 2,1,2007
85,13,1,ACTOPAN,1,2007
87,13,3,HUASTECA,1,2001
92,13,8,MOLANGO,1,1990
93,13,9,PACHUCA,1,1975
94,13,10,TENANGO,1,1989


Ahora vamos en que estado estan 

In [9]:
valores_entidad_a = pd.read_csv('Datos (CNGSPSPE) 2020/entidad_a_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
valores_entidad_a

,entidad_a,descrip
0,1,Aguascalientes
1,2,Baja California
2,3,Baja California Sur
3,4,Campeche
4,5,Coahuila de Zaragoza
5,6,Colima
6,7,Chiapas
7,8,Chihuahua
8,9,Ciudad de México
9,10,Durango


In [10]:
Df_CPS = Df_CPS.merge(
    valores_entidad_a[['entidad_a', 'descrip']],
    on='entidad_a',
    how='left'
)

In [13]:
Df_CPS['descrip'].value_counts()

descrip
Jalisco             12
Hidalgo              7
Quintana Roo         3
Durango              3
Ciudad de México     2
Oaxaca               2
Name: count, dtype: int64

In [14]:
Df_CPS

,entidad_a,numeral,nombre1,rensnn1,anio,descrip
0,9,11,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,1,2015,Ciudad de México
1,9,12,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,1,2015,Ciudad de México
2,10,1,CERESO NO. 1,1,1970,Durango
3,10,2,CE.DI.RE.SO. NO. 1,1,2001,Durango
4,10,3,CE.DI.RE.SO. NO. 2,1,2007,Durango
5,13,1,ACTOPAN,1,2007,Hidalgo
6,13,3,HUASTECA,1,2001,Hidalgo
7,13,8,MOLANGO,1,1990,Hidalgo
8,13,9,PACHUCA,1,1975,Hidalgo
9,13,10,TENANGO,1,1989,Hidalgo


In [19]:
Df_CPS.drop(columns=['entidad_a'], inplace=True)

Estos son los centros que se manejan bajo CPS

In [20]:
Df_CPS

,nombre1,anio,descrip
0,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,2015,Ciudad de México
1,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,2015,Ciudad de México
2,CERESO NO. 1,1970,Durango
3,CE.DI.RE.SO. NO. 1,2001,Durango
4,CE.DI.RE.SO. NO. 2,2007,Durango
5,ACTOPAN,2007,Hidalgo
6,HUASTECA,2001,Hidalgo
7,MOLANGO,1990,Hidalgo
8,PACHUCA,1975,Hidalgo
9,TENANGO,1989,Hidalgo


A partir de esto podemos ver el tipo de centro penitenciario, su clasificación o división del centro penitenciario y el modelo arquitectónico penitenciario

In [22]:
diccionarioP2= pd.read_csv('Datos (CNGSPSPE) 2020/diccionario_de_datos_m3s1p2_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
diccionarioP2

,COLUMNA,DESCRIPCION,ORIGEN,TIPO_DATO,LONGITUD,CODIGO_VALIDO
0,entidad_a,Entidad federativa,SECC I. 2,Numérico,2,1 al 99
1,numeral,Identificador o numeral del orden que guarda cada registro capturado dentro de la lista,SECC I. 2,Carácter,4,0-5000
2,nombre1,Nombre de los centros penitenciarios,SECC I. 2,Carácter,500,NaN
3,centpeni_a,Tipo de centro penitenciario,SECC I. 2,Numérico,2,1 al 99
4,clascent_a,Clasificación o división del centro penitenciario,SECC I. 2,Numérico,1,1 al 9
5,modarqui_a,Modelo arquitectónico penitenciario,SECC I. 2,Numérico,1,1 al 9


In [23]:
valores_centpeni_a = pd.read_csv('Datos (CNGSPSPE) 2020/centpeni_a_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
valores_centpeni_a

,centpeni_a,descrip
0,1,Centro de Readaptación Social para hombres
1,2,Centro de Readaptación Social para mujeres
2,3,Centro de Readaptación Social mixto
3,4,Reclusorio para hombres
4,5,Reclusorio para mujeres
5,6,Reclusorio mixto
6,7,Penitenciaría para hombres
7,8,Penitenciaría para mujeres
8,9,Penitenciaría mixta
9,10,Cárcel para hombres


In [25]:
valores_clascent_a = pd.read_csv('Datos (CNGSPSPE) 2020/clascent_a_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
valores_clascent_a

,clascent_a,descrip
0,1,Seguridad máxima
1,2,Seguridad media
2,3,Seguridad mínima
3,4,Hospital psiquiátrico y/o para infecciosos
4,5,Institución abierta
5,6,Sin clasificación
6,7,Otro
7,9,No se sabe


In [26]:
valores_modarqui_a = pd.read_csv('Datos (CNGSPSPE) 2020/modarqui_a_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
valores_modarqui_a	

,modarqui_a,descrip
0,1,Panóptico
1,2,Radial
2,3,Partido paralelo
3,4,Podular
4,5,Otro
5,9,No se sabe


In [28]:
df_p2= pd.read_csv('Datos (CNGSPSPE) 2020/m3s1p2_cngspspe2020.csv', 
                         encoding='utf-8-sig', 
                         low_memory=False
)
df_p2.head()

,entidad_a,numeral,nombre1,centpeni_a,clascent_a,modarqui_a
0,1,1,CENTRO DE REINSERCION SOCIAL AGUASCALIENTES,1,2,5
1,1,2,CENTRO DE REINSERCION SOCIAL EL LLANO,1,2,5
2,1,3,CENTRO DE REINSERCION SOCIAL PARA MUJERES,2,2,5
3,1,4,CENTRO DE REINSERCION SOCIAL DE MINIMA SEGURIDAD,3,5,5
4,2,1,CENTRO DE REINSERCIÓN SOCIAL MEXICALI,3,2,9


In [29]:
Df_CPS = Df_CPS.merge(
    df_p2[['nombre1', 'centpeni_a', 'clascent_a', 'modarqui_a']],
    on='nombre1',
    how='left'
)
Df_CPS

,nombre1,anio,descrip,centpeni_a,clascent_a,modarqui_a
0,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,2015,Ciudad de México,16,1,1
1,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,2015,Ciudad de México,16,1,1
2,CERESO NO. 1,1970,Durango,3,2,4
3,CE.DI.RE.SO. NO. 1,2001,Durango,1,2,3
4,CE.DI.RE.SO. NO. 2,2007,Durango,3,2,1
5,ACTOPAN,2007,Hidalgo,3,3,5
6,HUASTECA,2001,Hidalgo,3,2,5
7,MOLANGO,1990,Hidalgo,3,3,5
8,PACHUCA,1975,Hidalgo,3,3,5
9,TENANGO,1989,Hidalgo,3,3,5


Como ahora hay varias columnas por renombrar crearemos una funcion que lo haga 

In [30]:
def reemplazar_y_renombrar(df_principal, columna_codigo, df_valores, nuevo_nombre):
    # Crea el diccionario {código: descripción}
    mapeo = dict(zip(df_valores[columna_codigo], df_valores['descrip']))
    
    # Reemplaza los códigos por su significado
    df_principal[columna_codigo] = df_principal[columna_codigo].map(mapeo)
    
    # Renombra la columna a algo significativo
    df_principal = df_principal.rename(columns={columna_codigo: nuevo_nombre})
    
    return df_principal

In [31]:
Df_CPS = reemplazar_y_renombrar(Df_CPS, 'modarqui_a', valores_modarqui_a, 'Modelo arquitectónico penitenciario')
Df_CPS

,nombre1,anio,descrip,centpeni_a,clascent_a,Modelo arquitectónico penitenciario
0,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,2015,Ciudad de México,16,1,Panóptico
1,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,2015,Ciudad de México,16,1,Panóptico
2,CERESO NO. 1,1970,Durango,3,2,Podular
3,CE.DI.RE.SO. NO. 1,2001,Durango,1,2,Partido paralelo
4,CE.DI.RE.SO. NO. 2,2007,Durango,3,2,Panóptico
5,ACTOPAN,2007,Hidalgo,3,3,Otro
6,HUASTECA,2001,Hidalgo,3,2,Otro
7,MOLANGO,1990,Hidalgo,3,3,Otro
8,PACHUCA,1975,Hidalgo,3,3,Otro
9,TENANGO,1989,Hidalgo,3,3,Otro


In [32]:
Df_CPS = reemplazar_y_renombrar(Df_CPS, 'centpeni_a', valores_centpeni_a, 'Tipo de centro penitenciario')
Df_CPS = reemplazar_y_renombrar(Df_CPS, 'clascent_a', valores_clascent_a, 'Clasificación o división del centro penitenciario')
Df_CPS

,nombre1,anio,descrip,Tipo de centro penitenciario,Clasificación o división del centro penitenciario,Modelo arquitectónico penitenciario
0,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA I,2015,Ciudad de México,Centro de alta seguridad para delitos de alto impacto,Seguridad máxima,Panóptico
1,CENTRO VARONIL DE SEGURIDAD PENITENCIARIA II,2015,Ciudad de México,Centro de alta seguridad para delitos de alto impacto,Seguridad máxima,Panóptico
2,CERESO NO. 1,1970,Durango,Centro de Readaptación Social mixto,Seguridad media,Podular
3,CE.DI.RE.SO. NO. 1,2001,Durango,Centro de Readaptación Social para hombres,Seguridad media,Partido paralelo
4,CE.DI.RE.SO. NO. 2,2007,Durango,Centro de Readaptación Social mixto,Seguridad media,Panóptico
5,ACTOPAN,2007,Hidalgo,Centro de Readaptación Social mixto,Seguridad mínima,Otro
6,HUASTECA,2001,Hidalgo,Centro de Readaptación Social mixto,Seguridad media,Otro
7,MOLANGO,1990,Hidalgo,Centro de Readaptación Social mixto,Seguridad mínima,Otro
8,PACHUCA,1975,Hidalgo,Centro de Readaptación Social mixto,Seguridad mínima,Otro
9,TENANGO,1989,Hidalgo,Centro de Readaptación Social mixto,Seguridad mínima,Otro
